# COMP5329 — Deep Learning

**Tutorial — Multi-Modal Foundation Models**

**Semester 1, 2026**

### Learning Objectives
By the end of this tutorial you will be able to:
1. Explain how **Vision-Language Models (VLMs)** bridge image and text modalities, and classify the three main fusion paradigms (contrastive, projective, cross-attention).
2. Derive **cross-attention** from self-attention and state precisely which tensor supplies $Q$ and which tensor supplies $K,V$.
3. **Implement** a `CrossAttention` module from scratch — projections, scaled dot-product, optional padding mask, output projection.
4. Explain the **KV cache** optimisation for autoregressive decoding: why it exists, what it stores, and the prefill-vs-decode phase split.
5. **Implement** a `KVCache` class and a `decode_step` function, and demonstrate the compute saving over naive recomputation on a toy 8-step decode.
6. Answer exam-style short questions on cross-attention shapes, KV cache complexity, and how LLaVA's projector avoids needing an explicit cross-attention layer.


### Topic Coverage

Week 9 covers **multi-modal foundation models**. The full topic list (see `Week9_Self_Study.ipynb`) is:

- ✅ **Multimodal problem & VLM taxonomy** — token-level / feature-level / cross-attention fusion *(tutorial)*
- ✅ **Cross-attention from scratch** — $Q$ from one modality, $K,V$ from another, with padding mask *(tutorial, in-class coding)*
- ✅ **KV cache for autoregressive decoding** — incremental append, prefill vs decode, complexity analysis *(tutorial, in-class coding)*
- 📖 **CLIP (contrastive dual-tower)** — InfoNCE loss, zero-shot classification *(self-study)*
- 📖 **LLaVA (projector-based VLM)** — MLP connector, visual instruction tuning *(self-study)*
- 📖 **Flamingo (gated cross-attention VLM)** — Perceiver resampler, zero-initialised gate *(self-study)*
- 📖 **RLHF / DPO alignment** — preview; full coverage in **Week 10** *(self-study)*
- 📖 **Vision-Language-Action (VLA)** — RT-2, OpenVLA, actions as tokens *(self-study)*

Due to time constraints, the live tutorial **focuses on the two fundamental mechanisms** that make modern VLMs work: **cross-attention** (how modalities exchange information) and the **KV cache** (how autoregressive VLM inference is made fast). Concrete VLM architectures (CLIP, LLaVA, Flamingo) are only sketched at high level in Part A — their implementation is in the self-study notebook.

The live session is organised into three parts: **Part A** — tutor walkthrough, **Part B** — in-class coding exercise, **Part C** — exam-style Q&A.

---
# Part A · Tutor Review

> **Goal.** By the end of Part A you should be able to (a) state why naive self-attention cannot fuse two modalities, (b) place CLIP / LLaVA / Flamingo on a taxonomy of VLM fusion strategies, and (c) explain in one sentence why the autoregressive decoder of any VLM needs a KV cache.

---
## §0 — The Multimodal Problem

Everything we have built so far — BERT, GPT, ViT — consumes **one modality at a time**. But the real world is multimodal: an image rarely comes without a caption, a robot observation rarely comes without an instruction, a medical scan rarely comes without a report.

**The core question.** Given a vision encoder that outputs image features $V \in \mathbb{R}^{N_v \times d_v}$ and a language model that operates on text tokens $T \in \mathbb{R}^{N_t \times d_t}$, how do we make the two interact so the model can *jointly reason* over them?

There are essentially three answers, and every modern VLM is some mix of them.

| Fusion strategy | Where fusion happens | Example |
|---|---|---|
| **Feature-level (contrastive)** | In a shared embedding space *after* each encoder | **CLIP** |
| **Token-level (projection + concat)** | Image features are *projected* into the LLM's token space and prepended to text tokens | **LLaVA** |
| **Cross-attention (deep interleaving)** | New cross-attention layers inserted inside the LLM attend from text to vision | **Flamingo** |

Let us look at each in one short paragraph — **no code**, just the idea.

---
## §1 — Three VLM Paradigms (sketch only)

### 1.1 CLIP — Contrastive dual-tower

CLIP (Radford et al., 2021) trains **two separate encoders** — a ViT for images and a Transformer for text — to produce L2-normalised embeddings that live in the *same* $d$-dimensional space. The training objective is **symmetric InfoNCE**: in a batch of $N$ matched (image, caption) pairs, maximise the cosine similarity of diagonal pairs while minimising similarity of off-diagonal pairs.

$$\mathcal{L}_{\text{CLIP}} = -\tfrac{1}{2}\!\left[\sum_i \log \frac{e^{v_i^\top t_i / \tau}}{\sum_j e^{v_i^\top t_j / \tau}} + \sum_i \log \frac{e^{t_i^\top v_i / \tau}}{\sum_j e^{t_i^\top v_j / \tau}}\right]$$

The two towers **never exchange hidden states** — fusion is nothing more than a cosine similarity at the output. CLIP is therefore excellent at zero-shot retrieval and classification, but it **cannot generate text**.

### 1.2 LLaVA — Projector-based token-level fusion

LLaVA (Liu et al., 2023) takes a different route: use a frozen CLIP ViT to extract $N_v$ visual features, pass them through a **small MLP projector** that maps $\mathbb{R}^{d_v} \to \mathbb{R}^{d_{\text{llm}}}$, and then simply **concatenate** the resulting visual tokens with the text tokens before feeding the whole sequence into a standard LLM.

```
[IMG patch tokens after projector]  ++  [TEXT tokens]  →  LLM self-attention
```

The genius here is that **no new architecture is needed inside the LLM** — its existing self-attention handles vision-text interaction because, after projection, image features look *exactly like text tokens* to the model. This is why LLaVA is the simplest path from "LLM" to "VLM" and is the dominant open-source recipe today.

### 1.3 Flamingo — Gated cross-attention

Flamingo (Alayrac et al., 2022) keeps both the vision encoder and the LLM frozen, and inserts **new gated cross-attention layers** between the existing LLM blocks. Inside each inserted layer, the LLM's text hidden states act as the query $Q$, while a compressed set of visual tokens (produced by a Perceiver Resampler) provide $K$ and $V$:

$$\text{out} = x_{\text{text}} + \tanh(\alpha) \cdot \text{CrossAttn}(x_{\text{text}},\; x_{\text{visual}})$$

The learnable scalar gate $\alpha$ is **initialised to zero** so that $\tanh(\alpha)=0$ and the inserted layer is initially a no-op — the frozen LLM behaves exactly as before. During training the gate slowly opens, mixing in visual information without destroying the pretrained language priors.

### 1.4 Where this tutorial puts its effort

Two of the three paradigms above — LLaVA and Flamingo — depend directly on mechanisms this tutorial will cover *in depth*:

- Flamingo's gated layer is literally a **cross-attention** block (§B1) wrapped in a residual with a learnable gate.
- Every generative VLM (LLaVA, Flamingo, GPT-4V, Claude, Gemini) uses a **KV cache** (§B2) at inference time — without it, per-token latency would grow quadratically.

So we spend the rest of the tutorial making sure you can implement both from scratch.

> **Pointer — RLHF / DPO.** "How does a base VLM become a chat assistant?" is a *preference-alignment* question (SFT → RLHF → DPO). We cover it **in full next week (Week 10)**. For now, just know that every production VLM you have used has been through an alignment stage on top of the architecture we discuss here.

---
## §2 — Why Cross-Attention?

Self-attention relates tokens **within a single sequence**: $Q$, $K$, $V$ are all linear projections of the *same* input $X$. But if we want one sequence (text) to query another sequence (image patches), we need a mechanism where $Q$ is computed from one input and $K,V$ from *another*. That is cross-attention.

$$\text{SelfAttn}(X) = \text{softmax}\!\left(\frac{(XW_Q)(XW_K)^\top}{\sqrt{d_k}}\right) (XW_V)$$

$$\text{CrossAttn}(X_q, X_c) = \text{softmax}\!\left(\frac{(X_q W_Q)(X_c W_K)^\top}{\sqrt{d_k}}\right) (X_c W_V)$$

Two consequences worth pinning down before you start coding:

1. **The attention weight matrix is rectangular.** If $X_q \in \mathbb{R}^{T_q \times d}$ and $X_c \in \mathbb{R}^{T_c \times d}$ then $QK^\top \in \mathbb{R}^{T_q \times T_c}$ — it is *not* square in general.
2. **Masks behave differently.** In self-attention we often apply a *causal* mask over $(T, T)$. In cross-attention there is no causal relationship between $X_q$ and $X_c$; instead, we apply a **padding mask** over context positions, with shape $(B, 1, 1, T_c)$ so it broadcasts over heads and all query positions.

Both of these will come back as exam points.

---
## §3 — Why a KV Cache?

Autoregressive decoding generates token $x_t$ one at a time, conditioned on everything before it. A naive decoder recomputes attention over the full prefix $x_{1:t}$ at every step — a factor of $t$ of wasted work per step, and $O(T^3)$ over a full sequence of length $T$.

The **KV cache** removes this redundancy: we store every $K$ and $V$ we have already computed, so when the next token arrives we only need to

1. project the **single new token** to get $q_t, k_t, v_t$,
2. **append** $k_t, v_t$ to the cache, and
3. compute attention of the length-1 query against the length-$t$ cached keys/values.

The cache tensor has shape `[batch, heads, cached_len, d_head]` per layer, growing by exactly one entry along `cached_len` at each decode step. Two phases matter at inference time:

| Phase | Input | What happens | Cost |
|---|---|---|---|
| **Prefill** | The user prompt of length $T_p$, processed in **one parallel forward pass** | Populates the KV cache with $T_p$ entries | $O(T_p^2 d)$ (compute-bound) |
| **Decode** | One new token per step | Projects the new token, appends to cache, attends against all cached $K,V$ | $O(t d + d^2)$ per step (memory-bound) |

Without the cache, decode would cost $O(t^2 d)$ per step — dominated by a pointless recomputation of attention over the entire past. With the cache, decode is **linear** in the prefix length, not quadratic.

---
## §4 — Roadmap for Part B

In Part B you will fill in two small `nn.Module`s:

1. **`CrossAttention`** — three TODOs: project/reshape $Q, K, V$; apply an optional padding mask to the logits *before* softmax; output projection. Demo: batch 2, query length 4, context length 6, $d=16$.
2. **`KVCache` + `decode_step`** — three TODOs: append new $K,V$ to the cache; compute attention against the cached tensor; time a cached vs naive 8-step decode and print the speedup.

Every TODO has a matching solution cell with "Key points". Attempt the TODO yourself first — then expand the solution.


---
# Part B · In-Class Exercise

> **Your job**: fill in the `# TODO` blocks across the two tasks below. Each task has a collapsed **Solution** cell underneath — try the task yourself first, then expand the solution to compare.
>
> Both tasks use the same imports.


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)


## Task B1 · Cross-Attention from scratch (with optional padding mask)

Implement a multi-head **cross-attention** module. The module receives two sequences:

- a **query** sequence `x_query` of shape `(B, T_q, d_model)` — this feeds $Q$,
- a **memory / context** sequence `x_memory` of shape `(B, T_m, d_model)` — this feeds both $K$ and $V$.

It should optionally accept a padding mask `memory_mask` of shape `(B, T_m)` (1 = valid token, 0 = padding) and zero out attention to padded memory positions.

The attention weight matrix has shape `(B, num_heads, T_q, T_m)` — **rectangular**, not square.

### Contrast: Self-Attention vs Cross-Attention

| Aspect | Self-Attention | Cross-Attention |
|---|---|---|
| $Q$ comes from | the input $X$ | the **query** sequence $X_q$ |
| $K, V$ come from | the input $X$ | the **memory** sequence $X_m$ |
| Weight matrix shape | $(T, T)$ — square | $(T_q, T_m)$ — rectangular |
| Common mask | **Causal** mask on rows, to prevent peeking into the future | **Padding** mask on columns, to ignore padded memory tokens |
| Example | BERT, GPT internal layers | Encoder–decoder Transformer, Flamingo visual–text fusion |

**Three TODO steps** (all inside `forward`):
1. Project `x_query` → $Q$, `x_memory` → $K$, $V$; reshape to `(B, H, T, d_k)`.
2. Apply the padding mask to the attention logits **before softmax**: broadcast `(B, T_m)` to `(B, 1, 1, T_m)`, then `masked_fill(mask == 0, -inf)`.
3. Softmax, weighted sum with $V$, concat heads, apply output projection `W_o`.


In [ ]:
# ── Task B1: fill in the TODOs in the forward method ────────────────────
class CrossAttention(nn.Module):
    """Multi-head cross-attention.

    Q comes from `x_query`, K and V come from `x_memory`. An optional
    padding mask `memory_mask` of shape (B, T_m) marks valid memory positions.
    """

    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)   # queries come from x_query
        self.W_k = nn.Linear(d_model, d_model)   # keys   come from x_memory
        self.W_v = nn.Linear(d_model, d_model)   # values come from x_memory
        self.W_o = nn.Linear(d_model, d_model)   # output projection

    def forward(self, x_query, x_memory, memory_mask=None):
        B, T_q, _ = x_query.size()
        _, T_m, _ = x_memory.size()
        H, d_k = self.num_heads, self.d_k

        # TODO 1 — project and reshape. Target shapes:
        #   Q: (B, H, T_q, d_k)   from x_query
        #   K: (B, H, T_m, d_k)   from x_memory
        #   V: (B, H, T_m, d_k)   from x_memory
        Q = ...
        K = ...
        V = ...

        # Scaled dot-product logits: (B, H, T_q, T_m)  -- rectangular!
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

        # TODO 2 — apply the padding mask BEFORE softmax.
        #          memory_mask has shape (B, T_m); broadcast to (B, 1, 1, T_m)
        #          and set padded positions to -inf in `scores`.
        if memory_mask is not None:
            ...

        attn = F.softmax(scores, dim=-1)                      # (B, H, T_q, T_m)
        context = torch.matmul(attn, V)                        # (B, H, T_q, d_k)

        # Concatenate heads back to (B, T_q, d_model)
        context = context.transpose(1, 2).contiguous().view(B, T_q, self.d_model)

        # TODO 3 — apply the output projection W_o.
        out = ...

        return out, attn


<details>
<summary><b>▸ Solution · Task B1</b> (click to expand)</summary>

```python
def forward(self, x_query, x_memory, memory_mask=None):
    B, T_q, _ = x_query.size()
    _, T_m, _ = x_memory.size()
    H, d_k = self.num_heads, self.d_k

    # TODO 1 — project and reshape
    Q = self.W_q(x_query ).view(B, T_q, H, d_k).transpose(1, 2)   # (B, H, T_q, d_k)
    K = self.W_k(x_memory).view(B, T_m, H, d_k).transpose(1, 2)   # (B, H, T_m, d_k)
    V = self.W_v(x_memory).view(B, T_m, H, d_k).transpose(1, 2)   # (B, H, T_m, d_k)

    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)   # (B, H, T_q, T_m)

    # TODO 2 — padding mask before softmax
    if memory_mask is not None:
        mask = memory_mask.unsqueeze(1).unsqueeze(1)                 # (B, 1, 1, T_m)
        scores = scores.masked_fill(mask == 0, float("-inf"))

    attn    = F.softmax(scores, dim=-1)                              # (B, H, T_q, T_m)
    context = torch.matmul(attn, V)                                  # (B, H, T_q, d_k)
    context = context.transpose(1, 2).contiguous().view(B, T_q, self.d_model)

    # TODO 3 — output projection
    out = self.W_o(context)                                          # (B, T_q, d_model)
    return out, attn
```

**Key points:**
- **Which tensor feeds which projection.** $Q \leftarrow$ `x_query`, $K \leftarrow$ `x_memory`, $V \leftarrow$ `x_memory`. Mixing this up is the single most common bug — if you accidentally take $K$ from `x_query`, you have *rebuilt self-attention on the query sequence* and the memory is never read.
- **Rectangular weight matrix.** `scores` has shape `(B, H, T_q, T_m)` — note `T_q != T_m` in general. This is the signature of cross-attention; if you see a square weight matrix you are doing self-attention.
- **Mask before softmax, not after.** Setting padded logits to `-inf` guarantees that `softmax` assigns them exactly zero weight and *renormalises the unmasked weights to sum to 1*. Zeroing them after softmax would leave you with rows that sum to less than 1.
- **Mask broadcast shape.** Reshape `(B, T_m) → (B, 1, 1, T_m)` so the same mask is reused across heads and query positions without an explicit tile.
- **Output projection.** `W_o` re-mixes the concatenated head outputs — equivalent to the final dense layer in every multi-head attention block.
</details>


In [ ]:
# ── Task B1 demo: batch=2, query_len=4, memory_len=6, d=16, heads=4 ──
torch.manual_seed(0)
B, T_q, T_m, d, H = 2, 4, 6, 16, 4

x_query  = torch.randn(B, T_q, d)
x_memory = torch.randn(B, T_m, d)

# memory_mask: the first batch item has 4 valid memory tokens, the second has 6
memory_mask = torch.tensor([
    [1, 1, 1, 1, 0, 0],
    [1, 1, 1, 1, 1, 1],
])

ca = CrossAttention(d_model=d, num_heads=H)
out, attn = ca(x_query, x_memory, memory_mask=memory_mask)

print(f"x_query       : {tuple(x_query.shape)}   (B, T_q, d)")
print(f"x_memory      : {tuple(x_memory.shape)}   (B, T_m, d)")
print(f"output        : {tuple(out.shape)}        (B, T_q, d) — same length as query")
print(f"attention     : {tuple(attn.shape)}  (B, H, T_q, T_m) — rectangular")
print(f"row sums == 1 : {torch.allclose(attn.sum(dim=-1), torch.ones_like(attn.sum(dim=-1)))}")
print(f"padded weight : {attn[0, :, :, 4:].abs().max().item():.2e}  (should be ~0 for batch 0)")


---
## Task B2 · KV Cache for autoregressive decoding

Now we build a **KV cache** and a `decode_step` function that drives a tiny causal self-attention module one token at a time.

### Why a cache at all?

At decode step $t$, naive attention recomputes $K_{1:t}$ and $V_{1:t}$ from scratch — an $O(t)$ projection and an $O(t^2 d)$ attention. But $K_{1:t-1}$ and $V_{1:t-1}$ are *identical to what we already computed at step $t-1$*. That is pure waste. Summing over $t = 1, \dots, T$ the naive decoder spends $O(T^3 d)$ — cubic in sequence length — for no reason.

The KV cache stores every $K$ and $V$ already computed and appends one new entry per decode step.

| Quantity | Shape | Notes |
|---|---|---|
| `self.K` | `(B, H, cached_len, d_head)` | all previously seen keys |
| `self.V` | `(B, H, cached_len, d_head)` | all previously seen values |
| `new_k` | `(B, H, 1, d_head)` | just-projected key for the new token |
| `new_v` | `(B, H, 1, d_head)` | just-projected value for the new token |

### Prefill vs decode

Modern LLM serving splits inference into two phases:

1. **Prefill.** The user prompt of length $T_p$ is fed through the model in **one parallel forward pass**. The cache is populated with $T_p$ entries. Cost: $O(T_p^2 d)$ — compute-bound, saturates tensor cores.
2. **Decode.** For each generated token, we project only the new token, append to the cache, and attend the length-1 query against the length-$t$ cache. Cost per step: $O(t d + d^2)$ — memory-bandwidth-bound on real hardware because the cache must be streamed from HBM.

Without a cache, decode cost per step is $O(t^2 d)$, and total decode cost is $O(T^3 d)$ vs $O(T^2 d)$ **with** the cache — a $\Theta(T)$ speedup per token, turning a cubic blow-up into a quadratic one.

### Three TODO steps
1. **`KVCache.update`** — append `new_k, new_v` to `self.K, self.V` along the time axis. (If empty, initialise with them.)
2. **`CausalSelfAttentionCached.decode_step`** — push the newly projected $k, v$ into the cache, then compute attention of the length-1 query $q$ against the **full cached** $K, V$.
3. **Timing demo** — inside the demo cell, wrap the cached 8-step decode and the naive "re-run prefill over the whole prefix" baseline in `time.perf_counter()` calls and print the two wall-clock times so you can see the cached path being faster.


In [ ]:
# ── Task B2a: KVCache class — fill in update() ────────────────────────
class KVCache:
    """Ring-free KV cache: stores K, V tensors and appends along the time axis.

    Shapes:
        K, V : (B, H, cached_len, d_head)
    """

    def __init__(self):
        self.K = None        # filled on first update
        self.V = None

    def update(self, new_k: torch.Tensor, new_v: torch.Tensor):
        """Append new_k, new_v (each (B, H, 1, d_head)) to the cache.

        After calling:   self.K.shape == (B, H, cached_len + 1, d_head)
        """
        # TODO 1 — append along dim=2. If the cache is empty, initialise with new_k/new_v.
        if self.K is None:
            ...
        else:
            ...

    def get(self):
        """Return current (K, V) — (B, H, cached_len, d_head)."""
        return self.K, self.V

    def __len__(self):
        return 0 if self.K is None else self.K.size(2)


<details>
<summary><b>▸ Solution · Task B2 step 1 — KVCache.update</b> (click to expand)</summary>

```python
def update(self, new_k, new_v):
    if self.K is None:
        self.K = new_k
        self.V = new_v
    else:
        self.K = torch.cat([self.K, new_k], dim=2)   # time axis
        self.V = torch.cat([self.V, new_v], dim=2)
```

**Key points:**
- **Dim 2 is the time axis** once you have reshaped to `(B, H, T, d_head)`. Dim 0 is batch, dim 1 is heads, dim 3 is the head feature dimension — *none* of these grow during decoding.
- **First call is the special case.** The cache starts empty; the first `update` initialises it rather than concatenating to `None`. Forgetting this guard is a classic off-by-one bug.
- **Production caches are pre-allocated.** Real systems (vLLM, TensorRT-LLM) allocate `(B, H, T_max, d_head)` up front and maintain an explicit `cached_len` counter instead of growing with `torch.cat` — cat is fine for teaching but causes a full tensor realloc at every step.
</details>


In [ ]:
# ── Task B2b: a tiny causal self-attention that uses the cache ─────────
class CausalSelfAttentionCached(nn.Module):
    """Single-layer multi-head causal self-attention with an external KV cache.

    Two entry points:
        prefill(x_prompt)     — parallel forward pass over a whole prompt; fills cache.
        decode_step(x_new)    — one-token incremental forward pass using cache.
    """

    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    # ---- helper: reshape (B, T, d_model) -> (B, H, T, d_k) ----
    def _split(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.num_heads, self.d_k).transpose(1, 2)

    # ---- prefill: run the whole prompt in one pass, then seed the cache ----
    def prefill(self, x_prompt: torch.Tensor, cache: "KVCache"):
        B, T_p, _ = x_prompt.shape
        Q = self._split(self.W_q(x_prompt))                # (B, H, T_p, d_k)
        K = self._split(self.W_k(x_prompt))                # (B, H, T_p, d_k)
        V = self._split(self.W_v(x_prompt))                # (B, H, T_p, d_k)

        # Causal mask over the prompt only
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        causal = torch.tril(torch.ones(T_p, T_p, device=x_prompt.device))
        scores = scores.masked_fill(causal == 0, float("-inf"))
        attn = F.softmax(scores, dim=-1)
        out  = torch.matmul(attn, V)                        # (B, H, T_p, d_k)
        out  = out.transpose(1, 2).contiguous().view(B, T_p, self.d_model)

        # Seed the cache with every (K, V) we just computed (one big update)
        cache.update(K, V)
        return self.W_o(out)

    # ---- decode_step: one new token, uses the cache ----
    def decode_step(self, x_new: torch.Tensor, cache: "KVCache"):
        """
        Args:
            x_new: (B, 1, d_model) — embedding of the new token only
            cache: KVCache holding all previous K, V
        Returns:
            out: (B, 1, d_model) — attention output for the new token
        """
        B = x_new.size(0)

        # Project only the new token (this is the whole point of the cache!)
        q     = self._split(self.W_q(x_new))   # (B, H, 1, d_k)
        k_new = self._split(self.W_k(x_new))   # (B, H, 1, d_k)
        v_new = self._split(self.W_v(x_new))   # (B, H, 1, d_k)

        # TODO 2 — (i) push (k_new, v_new) into `cache`, then
        #          (ii) fetch the full cached K, V and compute attention of
        #               the length-1 query `q` against them.
        #          No causal mask needed — every cached entry is a past token.
        # Target shapes:
        #   K, V   : (B, H, cached_len, d_k)
        #   scores : (B, H, 1, cached_len)
        #   attn   : (B, H, 1, cached_len)
        #   out    : (B, H, 1, d_k)
        ...
        K, V   = ...
        scores = ...
        attn   = ...
        out    = ...

        out = out.transpose(1, 2).contiguous().view(B, 1, self.d_model)
        return self.W_o(out)


<details>
<summary><b>▸ Solution · Task B2 step 2 — decode_step</b> (click to expand)</summary>

```python
def decode_step(self, x_new, cache):
    B = x_new.size(0)

    q     = self._split(self.W_q(x_new))   # (B, H, 1, d_k)
    k_new = self._split(self.W_k(x_new))   # (B, H, 1, d_k)
    v_new = self._split(self.W_v(x_new))   # (B, H, 1, d_k)

    # TODO 2(i): append to cache
    cache.update(k_new, v_new)

    # TODO 2(ii): attend q against the full cached K, V
    K, V   = cache.get()                                                  # (B, H, t, d_k)
    scores = torch.matmul(q, K.transpose(-2, -1)) / math.sqrt(self.d_k)   # (B, H, 1, t)
    attn   = F.softmax(scores, dim=-1)
    out    = torch.matmul(attn, V)                                        # (B, H, 1, d_k)

    out = out.transpose(1, 2).contiguous().view(B, 1, self.d_model)
    return self.W_o(out)
```

**Key points:**
- **We only project the *new* token.** `W_q @ x_new`, `W_k @ x_new`, `W_v @ x_new` are each $O(d^2)$ — constant per decode step. Without a cache we would redo these projections for every past token at every step.
- **Query length is 1; key/value length is $t$.** `scores` has shape `(B, H, 1, t)`, so the softmax is over the *entire past*. This is a matrix–vector product on real hardware, which is why decode is memory-bandwidth-bound rather than compute-bound (see §3).
- **No causal mask is needed in decode.** Every entry in the cache corresponds to a *past* token by construction. Causal masking only matters during prefill, when many tokens are processed in parallel and we must stop each row from peeking forward.
- **The cache is a compute-for-memory trade.** Saving $O(T^2)$ recompute costs us $O(T)$ memory per layer. For a 7B model (32 layers, 32 heads, $d_k=128$) and $T=4096$ in fp16, the cache is roughly 1 GB — large, but much cheaper than the compute it replaces.
</details>


In [ ]:
# ── Task B2 demo: toy 8-step decode + cached vs naive compute comparison ──
torch.manual_seed(1)
d_model, num_heads = 32, 4
B = 1
T_prompt  = 4    # prefill length
T_decode  = 8    # how many tokens we decode autoregressively

layer = CausalSelfAttentionCached(d_model, num_heads)

# A toy stream of pre-generated token embeddings (prefill + decode)
prompt_emb = torch.randn(B, T_prompt, d_model)
new_embs   = torch.randn(B, T_decode, d_model)

# ── (A) Cached path ────────────────────────────────────────────────────────
cache = KVCache()
_ = layer.prefill(prompt_emb, cache)       # seeds cache with T_prompt entries
print(f"After prefill, cache length = {len(cache)}   (expected {T_prompt})")

# TODO 3 (i) — time the CACHED decode: 8 calls to layer.decode_step, each
#              using only the newest token (B, 1, d_model). Store elapsed
#              wall-clock seconds in `cached_time`.
t0 = time.perf_counter()
with torch.no_grad():
    for t in range(T_decode):
        x_new = new_embs[:, t:t+1]                 # (B, 1, d_model)
        ...                                        # <-- replace: one cached decode step
cached_time = time.perf_counter() - t0
print(f"After {T_decode} decode steps, cache length = {len(cache)}   "
      f"(expected {T_prompt + T_decode})")

# ── (B) Naive baseline ─────────────────────────────────────────────────────
# At every decode step we re-run `layer.prefill` over the FULL prefix from
# scratch — throwing away the cache each time. This is the "no-cache" cost.
# TODO 3 (ii) — time this naive path and store the elapsed seconds in
#               `naive_time`.
t0 = time.perf_counter()
with torch.no_grad():
    full = prompt_emb.clone()
    for t in range(T_decode):
        full = torch.cat([full, new_embs[:, t:t+1]], dim=1)   # grow sequence
        fresh_cache = KVCache()                               # thrown away each step
        ...                                                    # <-- replace: recompute prefill over `full`
naive_time = time.perf_counter() - t0

print()
print(f"Cached decode : {cached_time*1e3:7.2f} ms  (one projection per new token)")
print(f"Naive  decode : {naive_time *1e3:7.2f} ms  (re-project ALL past tokens per step)")
print(f"Speedup       : {naive_time/cached_time:5.2f}x on a toy 8-step decode")
print()
print("Complexity per decode step at time t:")
print("   Naive  :  O(t^2 * d)   — attention recomputed over the full prefix")
print("   Cached :  O(t   * d + d^2)   — only the new token is projected")


<details>
<summary><b>▸ Solution · Task B2 step 3 — timing demo</b> (click to expand)</summary>

```python
# TODO 3 (i) — cached path
t0 = time.perf_counter()
with torch.no_grad():
    for t in range(T_decode):
        x_new = new_embs[:, t:t+1]
        _ = layer.decode_step(x_new, cache)
cached_time = time.perf_counter() - t0

# TODO 3 (ii) — naive recompute path
t0 = time.perf_counter()
with torch.no_grad():
    full = prompt_emb.clone()
    for t in range(T_decode):
        full = torch.cat([full, new_embs[:, t:t+1]], dim=1)
        fresh_cache = KVCache()
        _ = layer.prefill(full, fresh_cache)
naive_time = time.perf_counter() - t0
```

**Key points:**
- **Why "re-run prefill" is the right naive baseline.** Without a cache, the attention at step $t$ depends on every token from $1$ through $t$, so we must project $K,V$ for every one of them — exactly what `prefill` does. Throwing away `fresh_cache` models the "no memoisation" world.
- **Why the speedup looks modest on toy sizes.** At $d=32$, $T=12$ the naive work is tiny; PyTorch framework overhead dominates. Try bumping `T_decode` to 64 or `d_model` to 256 and the ratio blows up — on real 7B models with $T=2048$, cached decoding is **orders of magnitude** faster per token.
- **Complexity recap.** Naive per-step is $O(t^2 d)$ (attention over length-$t$ prefix), cached per-step is $O(t d + d^2)$ (length-1 query over length-$t$ cache plus a constant-size projection). Summed over $T$ steps the totals are $O(T^3 d)$ vs $O(T^2 d + T d^2)$ — a $\Theta(T)$ per-token speedup and a quadratic overall saving.
- **What this demo does *not* measure.** On real hardware, decode is memory-bandwidth-bound because the cached $K,V$ must be streamed from HBM every step. The wall-clock speedup of a KV cache in production is dominated by that memory-bandwidth term, not by the FLOP count we are timing here.
</details>


---
# Part C · Exam-Style Questions

> Three short-answer questions at medium-high difficulty. Each has a collapsed **Model Answer** cell directly below with the key points the tutor will draw out during discussion.

## Q1 · Cross-attention vs self-attention — shapes and uses

**(a)** For a single multi-head attention block with $H$ heads and head dimension $d_k$, write down the shape of the attention weight tensor in both the self-attention case (sequence length $T$) and the cross-attention case (query length $T_q$, memory length $T_m$). State one architecturally significant difference between the two shapes.

**(b)** Give **one concrete example** of cross-attention where $T_q \ll T_m$ and **one** where $T_q \gg T_m$. In each case, identify which tensor is the query, which is the memory, and why the asymmetry makes sense.

**(c)** During training, self-attention inside a GPT decoder uses a **causal** mask, whereas cross-attention inside an encoder–decoder Transformer uses a **padding** mask on the memory side. Explain *why* these two masks are different — in particular, why it would be a bug to apply a causal mask to cross-attention, and why a padding mask alone would not suffice inside a GPT decoder.

<details>
<summary><b>▸ Model Answer · Q1</b></summary>

**(a)** Self-attention weight tensor: $(B, H, T, T)$ — **square**, one row and one column per sequence position. Cross-attention weight tensor: $(B, H, T_q, T_m)$ — **rectangular**, rows indexed by *query* positions and columns indexed by *memory* positions. The rectangular shape is the architectural fingerprint of cross-attention: there is no notion of "the diagonal" because queries and memory are not aligned position by position.

**(b)** $T_q \ll T_m$: an **image-captioning decoder** where a short text query of ~20 tokens attends over a long sequence of image patch features (e.g. $T_m = 256$ for a $16\times16$ ViT grid) — the short caption "looks at" the much larger image. The query is the text decoder state; the memory is the ViT output. $T_q \gg T_m$: a **LLM decoder reading a compressed knowledge vector**, such as Flamingo's Perceiver Resampler, which always emits exactly 64 learned visual tokens regardless of image size — here a long text context ($T_q = $ thousands) queries a small, fixed-size visual memory ($T_m = 64$). In both cases the query supplies the "what do I want to know?" signal and the memory supplies the "what is available to be looked up?".

**(c)** A causal mask enforces the rule "position $t$ may only attend to positions $\leq t$ within the *same* sequence" — it is about *time ordering inside one sequence*. A padding mask enforces the rule "some positions are not real tokens" — it is about *data validity across any sequence*.

- In **cross-attention**, query positions and memory positions are in different sequences; there is no temporal "before" relationship between them. Applying a causal mask would arbitrarily forbid query position 3 from looking at memory position 5 — which would be wrong, since the memory is *not* on the same timeline as the query. Hence cross-attention uses a padding mask on the memory side only.
- In a **GPT decoder**, self-attention operates on one sequence that may contain left-padding or right-padding (from batch packing). A padding mask on its own would let position 7 attend to position 9 if both were valid tokens — peeking at future text. So we need the causal mask *in addition* to any padding mask the batch happens to require.

**Key takeaway:** causal masks prevent *time-travel*; padding masks ignore *non-tokens*. They are orthogonal concerns, and which one applies where is determined by the architecture of the block, not by personal preference.
</details>


## Q2 · KV cache — complexity and validity

Consider a decoder-only Transformer with $d_{\text{model}} = 4096$, $L = 32$ layers, and a final generated sequence length of $T = 2048$ tokens. Assume $K$ and $V$ are stored per layer in `fp16` (2 bytes per scalar).

**(a)** Derive an expression in **big-O** notation for the cost of generating **one token at decode step $t$**, both *without* and *with* a KV cache. Sum both expressions over $t=1,\dots,T$ to get the total decode cost, and state the asymptotic ratio between the two.

**(b)** Compute the total KV cache memory footprint (in bytes, then gigabytes) for one sequence of length $T=2048$ at the settings above. Show your arithmetic. Why is batched long-context serving so expensive even when the model weights themselves easily fit in GPU memory?

**(c)** Name **two concrete situations** in which the KV cache you just built would become **invalid** and must be discarded or rebuilt. For each case, explain in one sentence *why* the cached $K,V$ no longer match the semantics of the current computation.

<details>
<summary><b>▸ Model Answer · Q2</b></summary>

**(a)** Without the cache, step $t$ must re-project all $t$ tokens to obtain $Q, K, V$ ($O(t d^2)$) and then run self-attention over the length-$t$ prefix ($O(t^2 d)$). The attention term dominates, so **one step = $O(t^2 d)$**. Summing over $t=1,\dots,T$:
$$\sum_{t=1}^{T} O(t^2 d) = O(T^3 d).$$
With the cache, step $t$ projects *only the new token* ($O(d^2)$) and attends its length-1 query against the length-$t$ cache ($O(t d)$). So **one step = $O(t d + d^2)$**, and summed:
$$\sum_{t=1}^{T} O(t d + d^2) = O(T^2 d + T d^2).$$
**Ratio.** Cached total / naive total $\sim O((T^2 d + T d^2) / (T^3 d)) = O(1/T + 1/(T^2/d)) \to 0$ as $T$ grows — the speedup is **$\Theta(T)$ per token**, turning a cubic cost into a quadratic one.

**(b)** Per layer we store both $K$ and $V$, each of shape $(T, d_{\text{model}})$. Per sequence:
$$\text{elements} = 2 \cdot L \cdot T \cdot d_{\text{model}} = 2 \cdot 32 \cdot 2048 \cdot 4096 \approx 5.37 \times 10^8$$
$$\text{bytes}    = 5.37 \times 10^8 \cdot 2 \approx 1.07 \times 10^9 \approx 1.0 \text{ GiB}.$$
This is **per sequence**, so serving a batch of 16 long-context requests on a 7B-parameter model easily consumes ~16 GiB just for the caches — more than the model weights themselves in fp16 (~14 GB). That is why the KV cache, not the parameters, is the memory bottleneck for long-context LLM serving, and why techniques like PagedAttention, MQA/GQA, and KV-cache quantisation exist.

**(c)** Two situations in which the cache becomes invalid:
1. **Out-of-order decoding or token rollback** (e.g. speculative decoding that rejects a draft, beam search pruning a hypothesis). The cache still holds $K,V$ for tokens that are no longer part of the current prefix, so attention would condition on a ghost past. Fix: truncate the cache back to the common prefix length before continuing.
2. **Any change in the context that sits "inside" the prefix** — editing a system prompt, changing a retrieval passage, switching user turn in an in-place multi-turn setup. The $K,V$ in the cache were computed from the *old* input embeddings and the *old* attention layers; if the tokens those entries represent changed, the cached keys/values no longer match the semantics of the current query.

*(Bonus third case, sometimes asked:* model-weight change — LoRA adapter swap, quantisation change. The $K,V$ tensors were computed by the old $W_k, W_v$, so they are stale.)

**Key takeaway:** a KV cache is a *memoisation* — it is only valid as long as the input sequence and the function computing $K,V$ both stay fixed. Any edit invalidates the memo.
</details>


## Q3 · Where is the cross-attention in LLaVA?

Part A contrasted LLaVA (projector + concat) with Flamingo (gated cross-attention layers). Both are VLMs, but only Flamingo has explicit cross-attention modules in its architecture.

**(a)** Draw the data-flow (in words is fine) of how an image and a text prompt pass through LLaVA — from raw pixels all the way to the first LLM self-attention layer. At which exact step do the visual features enter the LLM, and in what shape?

**(b)** LLaVA uses **no cross-attention module** anywhere. Explain, referring to the mechanics of self-attention on the mixed `[image_tokens ; text_tokens]` sequence, *why LLaVA does not need one*. Your answer should make clear what role the projector MLP plays and what role the LLM's existing self-attention plays.

**(c)** Flamingo, by contrast, inserts **gated cross-attention** layers into the frozen LLM. (i) Why is cross-attention the right mechanism here — i.e., why not just concatenate like LLaVA does? (ii) The gate $\tanh(\alpha)$ is initialised with $\alpha = 0$. Explain the purpose of this zero-initialisation trick, and what would go wrong if $\alpha$ were initialised to, say, $1$ instead.

<details>
<summary><b>▸ Model Answer · Q3</b></summary>

**(a)** LLaVA's forward pass, in order:
1. Image → frozen **CLIP ViT** → patch features of shape $(N_v, d_v)$, e.g. $(576, 1024)$ for a $336\times336$ image with $14\times14$ patches.
2. Patch features → **MLP projector** → projected visual tokens of shape $(N_v, d_{\text{llm}})$, e.g. $(576, 4096)$ for LLaMA-2.
3. Tokenise the text prompt → text embeddings of shape $(N_t, d_{\text{llm}})$.
4. **Concatenate** $[ \text{visual}_1, \dots, \text{visual}_{N_v},\, \text{text}_1, \dots, \text{text}_{N_t} ]$ into one sequence of length $N_v + N_t$ and dimension $d_{\text{llm}}$.
5. Feed this mixed sequence into the LLM's first self-attention layer exactly as if it were all text.

Visual features enter the LLM at **step 4**, in shape $(B, N_v + N_t, d_{\text{llm}})$ — indistinguishable from text at the tensor level.

**(b)** The projector MLP's one job is to **map CLIP's visual features into the LLM's token embedding space**. After this mapping, each visual token is a vector in $\mathbb{R}^{d_{\text{llm}}}$ drawn from roughly the same distribution as the LLM's own text-token embeddings. Once image and text live in the same vector space, **self-attention cannot tell them apart** — the attention mechanism only cares about pairwise dot products $q \cdot k$, not about where $q$ or $k$ came from. So self-attention on the mixed sequence naturally performs the fusion: whenever a text query says "describe the red cup", its $Q$ vector develops a high inner product with whichever visual $K$ vector represents the red cup, and the corresponding $V$ (also visual) is read back. Cross-attention is unnecessary because we turned the vision–text interaction into a *self*-attention problem by first aligning the two spaces with the projector.

**Role summary:**
- **Projector MLP** — makes vision and text *commensurate* in a shared space (the hard part, learned on (image, caption) data).
- **LLM self-attention** — performs the *fusion* for free, because after projection there is only one sequence, not two.

**(c)** **(i) Why cross-attention in Flamingo.** Flamingo was specifically designed to **keep both the vision encoder and the LLM frozen** and insert new layers between them. Concatenation à la LLaVA requires the text-token and image-token spaces to be compatible, which in turn requires training a projector *plus* (in practice) fine-tuning the LLM on visual instruction data. Flamingo instead leaves the LLM weights untouched and uses cross-attention to let the frozen text hidden states *query* a compressed set of visual tokens. That way the visual information is read on-demand by each text position, no pre-alignment of spaces is needed, and the LLM's language abilities are preserved exactly. Cross-attention is the natural fit because $Q$ and $K,V$ genuinely come from different sources here.

**(ii) Why $\tanh(0) = 0$.** At initialisation, the newly inserted cross-attention layer's weights are essentially random. If you used the raw cross-attention output as a residual addition to the frozen LLM's hidden state, the LLM would be **flooded with random junk from the first training step**, destabilising or overwriting the pretrained language representations before the new weights have learned anything useful. Initialising $\alpha = 0$ makes the gated residual $x + \tanh(0)\cdot\text{CrossAttn}(x, v) = x$ — the inserted layer is mathematically a **no-op** at $t=0$, so the frozen LLM behaves exactly as before. As training progresses, $\alpha$ slowly grows and visual information bleeds in smoothly, but only after the new cross-attention weights have stabilised. Initialising $\alpha=1$ would skip this warm-up and is known to degrade or destroy the pretrained LLM behaviour.

*(This zero-init trick recurs throughout modern DL: LoRA initialises its second low-rank matrix $B$ to zero for the same reason, and ControlNet uses "zero convolutions" — both preserve the pretrained function at $t=0$.)*
</details>

---

*End of Part C.*


---
## Summary

| Section | Key concept |
|---|---|
| **§0–§1** | Multimodal problem → three VLM fusion paradigms (contrastive / projector / cross-attention) → CLIP, LLaVA, Flamingo in one paragraph each |
| **§2** | Cross-attention: $Q$ from one sequence, $K,V$ from another; weight matrix $(T_q, T_m)$ rectangular; padding mask on memory, *not* causal |
| **§3** | KV cache: store per-layer $(B, H, \text{cached\_len}, d_k)$; prefill (parallel, compute-bound) vs decode (incremental, memory-bound); $O(T^3)\!\to\!O(T^2)$ |
| **B1** | `CrossAttention` from scratch — projection, rectangular scores, padding mask before softmax, output projection |
| **B2** | `KVCache` + `decode_step` — append new $K,V$, attend length-1 query against cache, $\Theta(T)$ speedup over naive recompute |

**Take-aways:**
- Cross-attention is the single mechanism that lets "one sequence query another" — it underpins every VLM fusion strategy that deserves the name "deep fusion".
- The KV cache is not an optimisation detail — it is the difference between quadratic and cubic inference, and the reason why long-context LLM serving is memory-bound rather than compute-bound.
- LLaVA does not need cross-attention because its projector MLP *first* aligns the two modalities into a shared token space; after that, the LLM's self-attention handles fusion for free. Flamingo needs cross-attention because it keeps both the vision encoder and the LLM frozen and must route visual information through newly inserted layers.
- **Next week (Week 10)** we pick up the alignment story — SFT, RLHF, DPO — that turns a base VLM into a helpful assistant.
